<a href="https://colab.research.google.com/github/pastrop/kaggle/blob/master/RAG_experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install llama-index llama-index-llms-anthropic llama-index-embeddings-huggingface anthropic

In [ ]:
"""
RAG with LlamaIndex + Claude + local embeddings.
- LLM:        Claude Sonnet 4.6 (Anthropic API) — used to synthesize answers
- Embeddings: BAAI/bge-small-en-v1.5 (local, free, runs on CPU)
- Loader:     SimpleDirectoryReader (handles PDF + Markdown)
- Citations:  CitationQueryEngine (inline [1][2] markers + source metadata)
"""

from __future__ import annotations

import os
import sys
from pathlib import Path

import anthropic
from llama_index.core import Settings, SimpleDirectoryReader, VectorStoreIndex
from llama_index.core.query_engine import CitationQueryEngine
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.anthropic import Anthropic

# ---------- Config ----------
DOCS_DIR = "/content"                          # where your PDFs/MDs live
LLM_MODEL = "claude-sonnet-4-5"              # solid balance of cost/quality
EMBED_MODEL = "BAAI/bge-small-en-v1.5"       # local, free
TOP_K = 3                                    # how many chunks to retrieve


In [ ]:
# ---------- 1. Validate the API key ----------
def validate_api_key() -> str:
    """Fail fast with a clear message if the key is missing, malformed, or invalid."""
    api_key = os.getenv("ANTHROPIC_API_KEY")

    if not api_key:
        sys.exit(
            "ERROR: ANTHROPIC_API_KEY is not set.\n"
            "  1. Get a key at https://console.anthropic.com/\n"
            "  2. Set it:  export ANTHROPIC_API_KEY='sk-ant-...'"
        )

    if not api_key.startswith("sk-ant-"):
        print(
            "WARNING: key doesn't start with 'sk-ant-'. "
            "Anthropic keys normally do. Continuing anyway..."
        )

    # Lightweight liveness check: smallest possible call to confirm the key works.
    # Costs a fraction of a cent and surfaces auth errors immediately instead of
    # after you've built the entire index.
    try:
        client = anthropic.Anthropic(api_key=api_key)
        client.messages.create(
            model="claude-haiku-4-5",  # cheapest model for the probe
            max_tokens=1,
            messages=[{"role": "user", "content": "ping"}],
        )
    except anthropic.AuthenticationError:
        sys.exit("ERROR: ANTHROPIC_API_KEY is invalid or has been revoked.")
    except anthropic.RateLimitError:
        print("WARNING: hit a rate limit on the key check — key is valid, continuing.")
    except anthropic.APIConnectionError as e:
        sys.exit(f"ERROR: could not reach Anthropic API: {e}")
    except Exception as e:
        # Non-fatal: don't block startup on unexpected probe failures
        print(f"WARNING: key liveness check failed unexpectedly: {e}")

    print("✓ Anthropic API key validated.")
    return api_key

In [ ]:
# ---------- 2. Configure LlamaIndex globally ----------
def configure(api_key: str) -> None:
    Settings.llm = Anthropic(
        model=LLM_MODEL,
        api_key=api_key,
        max_tokens=1024,
    )
    # First run downloads the model (~133 MB) to ~/.cache/huggingface/
    Settings.embed_model = HuggingFaceEmbedding(model_name=EMBED_MODEL)
    print(f"✓ Configured: LLM={LLM_MODEL}, embeddings={EMBED_MODEL} (local)")


# ---------- 3. Load documents and build the index ----------
def build_index(docs_dir: str) -> VectorStoreIndex:
    path = Path(docs_dir)
    if not path.exists():
        sys.exit(f"ERROR: documents directory '{docs_dir}' not found.")

    documents = SimpleDirectoryReader(
        input_dir=docs_dir,
        required_exts=[".pdf", ".md"],
        recursive=True,
        filename_as_id=True,    # stable IDs based on filename
    ).load_data()

    if not documents:
        sys.exit(f"ERROR: no .pdf or .md files found in '{docs_dir}'.")

    print(f"✓ Loaded {len(documents)} document chunk(s). Building index...")
    # All embedding happens locally here — no API calls.
    return VectorStoreIndex.from_documents(documents, show_progress=True)


# ---------- 4. Query with inline citations ----------
def ask(index: VectorStoreIndex, question: str) -> None:
    query_engine = CitationQueryEngine.from_args(
        index,
        similarity_top_k=TOP_K,
        citation_chunk_size=512,
    )
    response = query_engine.query(question)

    print(f"\n=== Q: {question} ===")
    print(f"\nANSWER:\n{response.response}")

    print("\nSOURCES:")
    for i, node in enumerate(response.source_nodes, 1):
        meta = node.metadata
        fname = meta.get("file_name", "unknown")
        page = meta.get("page_label", "—")  # PDFs only; MDs won't have this
        score = node.score if node.score is not None else 0.0
        print(f"  [{i}] {fname}  (page: {page})  score={score:.3f}")

In [ ]:
key = validate_api_key()

In [ ]:
configure(key)
index = build_index(DOCS_DIR)

In [ ]:
ask(index, "What are the data retention policies?")
ask(index, "Who is responsible for incident response?")